# Mamba SOH — LFP chemistry variant (GH-67 Mức 2)

Retrain kiến trúc `MambaSOHPredictor` (window=30, d_model=64, d_state=16) trên dataset
**Severson et al. 2019** (Nature Energy) — LFP/graphite A123 APR18650M1A, 1.1 Ah — để có bộ
artifact riêng cho chemistry LFP. **KHÔNG đụng** model NASA/NMC production (`soh_mamba_v1.6.pth`).

---

## Checklist trước khi Run All

| # | Việc | Ghi chú |
|---|------|---------|
| 1 | Settings → Accelerator → **GPU T4 x2** | KHÔNG chọn P100 — PyTorch Kaggle đã bỏ sm_60 |
| 2 | **+ Add Data** → dataset chứa `.mat` Severson | Batch1/2/3(/4) từ https://data.matr.io/1/ |
| 3 | Add-ons → Secrets → `GITHUB_TOKEN` | GitHub PAT (nếu repo private) |
| 4 | **`git push` code mới lên GitHub TRƯỚC** | Cell 3 sẽ kiểm tra và dừng nếu thiếu |

> ⚠️ **Mục 4 là chỗ dễ mất thời gian nhất.** Notebook clone code từ GitHub remote — sửa file
> trên máy local mà chưa push thì Kaggle **không thấy**. Lần chạy 2026-07-25 mất ~11 giờ và ra
> kết quả y hệt lần trước vì lý do này. Cell 3 giờ tự chặn trường hợp đó.

## Lịch sử kết quả

| Lần | Cấu hình | Test MAE | Test RMSE | Đạt target |
|-----|----------|----------|-----------|------------|
| 1 | `--epochs 5`, `CYCLE_COUNT_NORM=200` | 2.5856% | 3.4745% | ❌ |
| 2 | `--epochs 100`, `CYCLE_COUNT_NORM=2300` | 1.9213% | 2.7627% | ✅ |
| 3 | + chỉ pha xả + lọc outlier phi vật lý | ? | ? | — |

Target: **MAE < 2.0%** · **RMSE < 3.0%** · Tham chiếu NASA v1.6 cùng kiến trúc: 1.34% / 1.84%

Lần 3 sửa 2 lỗi **đúng đắn** (không phải tuning): windows lẫn pha sạc, và outlier phi vật lý
làm hỏng scaler. Nếu vẫn muốn ép thêm sau lần 3, `train.py` còn 3 knob chưa dùng cho path
window=30: `--swa`, `--jitter 0.01`, `--balance-bands` (xem §7).

Output: `soh_mamba_v2.0-lfp.pth`, `isolation_forest_v2.0-lfp.pkl`, `scaler_lfp.pkl`,
`feature_scaler_lfp.pkl` — 4 file riêng, không ghi đè artifact NASA.

## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chua bat GPU: Settings -> Accelerator -> GPU T4 x2'
name = torch.cuda.get_device_name(0)
print('GPU:', name)
assert 'P100' not in name, 'P100 khong tuong thich PyTorch Kaggle (sm_60) - doi sang GPU T4 x2'

## 2 — Clone repo

In [ ]:
import subprocess, os
BRANCH = 'feat/GH-67-lfp-retrain-severson'
REPO   = '/kaggle/working/ai-module'
url = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('Khong co GITHUB_TOKEN secret -> thu public clone:', e)

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, REPO], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git', 'log', '-1', '--format=%h %ci %s']).decode())

## 3 — Kiểm tra code clone về có đúng bản mới không

Chặn đúng cái bẫy đã làm mất ~11 giờ: notebook đã sửa nhưng code trên GitHub thì chưa.
Nếu cell này fail → về máy chạy `git push` rồi **Restart & Run All** (phải xoá session để
clone lại, vì cell 2 skip clone khi thư mục đã tồn tại).

In [ ]:
import pathlib

checks = {
    'scripts/preprocess_lfp.py': ['--cycle-stride', 'PHYSICAL_RANGES', '_nonphysical_channel',
                                  '--phase', '_longest_discharge_segment'],
    'scripts/train.py':          ['--feature-scaler-version', '--mamba-out', '--iso-out'],
    'src/core/config.py':        ['LFP_CYCLE_COUNT_NORM', 'LFP_NOMINAL_CAPACITY_AH'],
}
missing = []
for path, needles in checks.items():
    text = pathlib.Path(path).read_text(encoding='utf-8')
    for n in needles:
        status = 'OK  ' if n in text else 'THIEU'
        print(f'  [{status}] {path}: {n}')
        if n not in text:
            missing.append(f'{path}::{n}')

assert not missing, (
    'Code clone ve THIEU cac fix sau: ' + ', '.join(missing) +
    '\n-> Ve may chay: git add -A && git commit && git push, roi Restart & Run All.'
)
print('\nTat ca fix da co trong code clone ve.')

## 4 — Dependencies

In [ ]:
%pip install -q h5py scipy scikit-learn joblib pandas
import h5py, scipy, sklearn
print('h5py', h5py.__version__, '| scipy', scipy.__version__, '| sklearn', sklearn.__version__)

## 5 — Tìm dataset Severson

Cần ít nhất 1 file `*.mat` có chữ "batch" trong tên (vd
`2017-05-12_batchdata_updated_struct_errorcorrect.mat`). Fail → kiểm tra lại **+ Add Data**.

In [ ]:
import os, subprocess
found = [f for f in subprocess.check_output(
    ['find', '/kaggle/input', '-iname', '*batch*.mat']).decode().splitlines() if f]
assert found, 'Khong thay file *batch*.mat - dung + Add Data de attach dataset Severson truoc'
DATASET = os.path.dirname(found[0])
os.chdir('/kaggle/working/ai-module')
print('DATASET:', DATASET)
for f in found:
    print('  ', f)

## 6 — Preprocess (Severson `.mat` → window=30, 6 feature)

Hai fix **đúng đắn** (không phải tuning) áp dụng ở bước này, mặc định đã bật:

- **`--phase discharge`** — chỉ lấy đoạn **xả** dài nhất của mỗi cycle. Severson lưu nguyên
  cycle gồm cả sạc nhanh nhiều bước (dòng dương tới ~8 A) lẫn xả 4C, trong khi NASA
  (`scripts/preprocess.py`) chỉ nạp chu kỳ xả, và inference cũng chỉ thấy telemetry xả.
  Trước fix này ~nửa số window là pha **sạc** — model phải đoán nhãn dung lượng *xả*
  (`QDischarge`) từ mẫu sạc.
- **`PHYSICAL_RANGES`** — drop cycle chứa giá trị sensor phi vật lý.

**Dùng toàn bộ cycle (không cắt).** Lần chạy trước chứng minh full data vẫn xong trong 12h,
nên cắt bớt chỉ làm mất dữ liệu train. Nếu session sắp hết giờ, thêm `--cycle-stride 5`.

### Đọc 4 thứ trong log

1. **`scaler range temperature`** — kỳ vọng ~`[25, 50]`. Nếu vẫn `[-270, 400]` → filter chưa ăn.
2. **`scaler range time`** — kỳ vọng bắt đầu từ `0.000`, không còn số âm.
3. **`discharge duration trung vi`** — script tự đoán đơn vị thời gian. Nếu in ra
   `PHUT? -> soc_percent lech 60x, BAO LAI` thì **dừng lại báo tôi**: `compute_soc_percent()`
   giả định giây, kênh `soc_percent` sẽ sai 60×.
4. **Số cycle bị drop** — vài chục là bình thường. Drop tỉ lệ lớn → đang vứt dữ liệu thật.
   Riêng `no discharge run >= 30` nhiều bất thường → ngưỡng phát hiện pha cần xem lại.

Ngoài ra `[TIMING]` tách thời gian parse `.mat` vs trích xuất window/feature + số step/epoch.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess_lfp.py --data-dir "{DATASET}" --output-dir data/processed_lfp

## 7 — Train

Kiến trúc + hyperparameter **giữ nguyên** như model NASA. `--mamba-out`/`--iso-out`/
`--model-version` đảm bảo KHÔNG ghi đè `soh_mamba_v1.6.pth` production.

`train.py` chỉ in dòng metric khi `epoch % 10 == 0` → sẽ có 10 dòng (epoch 10, 20, …, 100).
So `TrainLoss` với `ValLoss`:

- cả 2 còn cao và đang giảm → vẫn underfit, cần thêm epoch
- TrainLoss thấp mà ValLoss cao → overfit, cần regularization
- cả 2 phẳng sớm → đã hội tụ; MAE miss target là do dữ liệu/kiến trúc, không phải số epoch

### Nếu muốn ép thêm ở lần chạy SAU

`train()` cho window=30 chỉ nhận 4 knob: `epochs`, `--balance-bands`, `--jitter`, `--swa`
(mọi flag khác như `--pooling`/`--patch-size`/`--dropout` chỉ dùng cho `--long`). Lần 2 không
bật cái nào. **Đừng bật cùng lúc với 2 fix ở §6** — chạy §6 trước để biết fix đúng đắn đóng góp
bao nhiêu, rồi mới thêm từng knob:

| Knob | Kỳ vọng | Rủi ro |
|------|---------|--------|
| `--swa` | Trung bình hoá trọng số 25% epoch cuối → val/test ổn định hơn | Rất thấp, không tốn thêm epoch |
| `--jitter 0.01` | Nhiễu Gauss lên input, chống overfit (127 cell, window rất tương quan) | Quá cao sẽ làm underfit |
| `--balance-bands` | Cân bằng theo lưới (nhiệt độ × SOH) | **Yếu với LFP** — xem ghi chú dưới |

> `--balance-bands` chia SOH thành 10 bin trên thang [0,100]; dữ liệu Severson chỉ trải
> ~80–100% nên rơi vào đúng **2 bin**, và Severson chỉ có 1 mức nhiệt (30°C) nên chiều nhiệt độ
> của lưới cũng sụp. Nghĩa là nó gần như không làm gì — khác hẳn NASA (3 mức nhiệt, SOH trải
> rộng hơn) nơi flag này sinh ra. Đừng kỳ vọng nhiều.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py \
    --data-dir data/processed_lfp \
    --epochs 100 \
    --log-dir logs/training \
    --mamba-out models/weights/soh_mamba_v2.0-lfp.pth \
    --iso-out models/weights/isolation_forest_v2.0-lfp.pkl \
    --model-version 2.0-lfp \
    --feature-scaler-version 2.0-lfp

## 8 — Kiểm tra target + so với lần trước

In [ ]:
import torch, joblib

PREV_MAE, PREV_RMSE = 1.9213, 2.7627   # lan 2 (epochs=100, chua loc outlier)
TARGET_MAE, TARGET_RMSE = 2.0, 3.0

ck = torch.load('models/weights/soh_mamba_v2.0-lfp.pth', map_location='cpu', weights_only=False)
mae, rmse = ck['test_mae'], ck['test_rmse']

print(f"Test MAE : {mae:.4f}%  (target < {TARGET_MAE})   lan truoc {PREV_MAE}  -> {PREV_MAE - mae:+.4f}")
print(f"Test RMSE: {rmse:.4f}%  (target < {TARGET_RMSE})   lan truoc {PREV_RMSE}  -> {PREV_RMSE - rmse:+.4f}")
print('DAT TARGET:', mae < TARGET_MAE and rmse < TARGET_RMSE)

# Xac nhan filter outlier da an: dai scaler phai hop ly, khong con sentinel
s = joblib.load('models/weights/scaler_lfp.pkl')
sc = s['scaler']
print('\nDai scaler (phai hop ly, khong con -270/400):')
for i, n in enumerate(['voltage', 'current', 'temperature']):
    print(f'  {n:<12}: [{sc.data_min_[i]:9.3f}, {sc.data_max_[i]:9.3f}]')
print('\nMetadata scaler:', {k: v for k, v in s.items() if k not in ('scaler', 'trained_on')})
print('So cell train:', len(s['trained_on']))

## 9 — Đóng gói artifact để tải về

4 file cần copy vào `models/weights/` trên máy. **Không tự commit** — tải zip về, tự
`git add` + `git commit` + `git push` theo quy trình repo.

In [ ]:
import shutil, os
OUT = '/kaggle/working/lfp_artifacts'
os.makedirs(OUT, exist_ok=True)
for p in [
    'models/weights/soh_mamba_v2.0-lfp.pth',
    'models/weights/isolation_forest_v2.0-lfp.pkl',
    'models/weights/scaler_lfp.pkl',
    'models/weights/feature_scaler_lfp.pkl',
]:
    shutil.copy(p, OUT)
    print('  +', os.path.basename(p), f'({os.path.getsize(p)/1024:.0f} KB)')
shutil.make_archive(OUT, 'zip', OUT)
print('\nTai ve: lfp_artifacts.zip (tab Output ben phai)')

## 10 — Dọn output (xoá repo clone — artifact đã nằm trong zip)

In [ ]:
import os, shutil
os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/ai-module', ignore_errors=True)
shutil.rmtree('/kaggle/working/lfp_artifacts', ignore_errors=True)
print('Output con lai:', sorted(os.listdir('/kaggle/working')))